In [1]:
import json
import os
import requests
import time

In [ ]:
rename_map = {
    "Douglas": "Douglas (Isle of Man)",
    "North Nicosia": "Nicosia (North)"
}

In [3]:
def get_cities_from_ontology(ontology_path):
    with open(ontology_path) as f:
        content = json.loads(f.read())
    cities = content.keys()
    return cities

In [ ]:
BASE_PATH = "/kaggle/input/datasets/alessandroisceri/"
OUT_BASE_PATH = "/kaggle/working/"
eu_cities = get_cities_from_ontology(BASE_PATH + "europe-ontology/europe_ontology_v5.json")
print(len(eu_cities))

134


In [5]:
def create_txt_file(folder, city, page_content):
    city = city.replace("'", "")
    with open(f"{folder}/{city}.txt", "w", encoding="utf-8") as file:
        file.write(page_content)

In [6]:
headers = {"User-Agent": "city-script/1.0"}
params = {
    "action": "query",
    "prop": "extracts",
    "format": "json",
    "explaintext": 1
}

def send_request(url, city):
    params["titles"] = city
    res = requests.get(url, params=params, headers=headers)
    res_json = res.json()
    page_id = list(res_json["query"]["pages"].keys())[0]
    page_res_json = res_json["query"]["pages"][page_id]
    return page_res_json

In [ ]:
url_en = "https://en.wikivoyage.org/w/api.php"

def get_cities_page_content(cities, folder):

    for city in cities:

        city_API = city
        if city in rename_map:
            city_API = rename_map[city]
            
        page_res_json = send_request(url_en, city_API)
    
        if "extract" in page_res_json:
            create_txt_file(folder, city, page_res_json["extract"])
                
        time.sleep(0.5)

In [8]:
os.makedirs(OUT_BASE_PATH + "eu/")
get_cities_page_content(eu_cities, "eu/")